# 13 — LLM-as-judge ненадёжен → гибрид (правила + LLM)

> **Проблема:** «спроси LLM есть ли SQLi» не работает надёжно.
> **Решение (ADR-0004):** Phase 1 (детерминированные правила) выдаёт
> findings → Phase 2 (LLM) делает **триаж** + объяснение с RAG-ссылками.

## Что покажем

На 4 кейсах сравним:
- **A.** Только Phase 1 (алгоритм) → видим FP и FN.
- **B.** Только Phase 2 (LLM-only) → видим inconsistency.
- **C.** Гибрид (Phase 1 → Phase 2 триаж) → FP отсеяны, объяснения с CWE.

Все «вызовы LLM» — mock-функции.


## 🧒 Аналогия для ребёнка

Идёт спортивный турнир. Есть два судьи:
- **Робот-судья** с правилами «если игрок упал → красная карточка».
  Точный, но **тупой**: упал по своей инициативе → красная.
  Не упал в очевидном фоле → ничего.
- **Человек-судья** смотрит контекст: «упал, но симулировал → нет
  карточки. Не упал, но фол был → жёлтая». Гибче, но **устаёт**
  и иногда ошибается, забывает правила.

**Гибрид:** робот размечает «формальные нарушения», человек
делает финальное решение с учётом контекста. Это и есть Phase 1
+ Phase 2.


## 1. Setup — 4 кейса


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief 4 кейса для сравнения судей.
# @details
#   FP — false positive (алгоритм кричит, на самом деле всё ок).
#   FN — false negative (алгоритм молчит, реально уязвимо).
#   TP — true positive (всё корректно).
#   TN — true negative.
CASES = [
    {
        "id": "case-1-legitimate-pg-sleep",
        "sql": "-- миграция, пауза для autovacuum\nALTER TABLE big SET (autovacuum_vacuum_scale_factor = 0.01);\nSELECT pg_sleep(1);",
        "ground_truth_vuln": False,
        "context_note": "DDL-миграция, нет user input, pg_sleep ради autovacuum",
        "kind": "FP-test (тест на ложное срабатывание)",
    },
    {
        "id": "case-2-blind-injection",
        "sql": "UPDATE last_seen SET ts=now() WHERE user_id = 1 OR (CASE WHEN substr((SELECT password FROM users WHERE id=1),1,1)='a' THEN pg_sleep(2) ELSE 0 END)",
        "ground_truth_vuln": True,
        "context_note": "blind exfil через time-based CASE",
        "kind": "TP-test (надо поймать)",
    },
    {
        "id": "case-3-semantic-leak",
        "sql": "SELECT u.id, u.login, r.permissions FROM users u JOIN role_assignments ra ON ra.user_id=u.id JOIN roles r ON r.id=ra.role_id",
        "ground_truth_vuln": True,
        "context_note": "permissions = JSON прав доступа; имя колонки не матчит regex sensitive",
        "kind": "FN-test (формально ок, по смыслу — leak)",
    },
    {
        "id": "case-4-safe-aggregate",
        "sql": "SELECT COUNT(*) FROM clients WHERE balance > 100000",
        "ground_truth_vuln": False,
        "context_note": "агрегат, ничего не утекает",
        "kind": "TN-test (молчать)",
    },
]

for c in CASES:
    print(f"  [{c['kind']:35s}] {c['id']}")


## 2. Judge A — только Phase 1 (детерминированный)


In [ ]:
##
# @brief Phase 1 — простой rule-based детектор.
def judge_A_rules_only(sql):
    findings = []
    if re.search(r"\bpg_sleep\s*\(", sql, re.IGNORECASE):
        findings.append({"rule_id": "R006-pg-sleep", "vuln_class": "SQL_INJ_TIME", "risk": 8})
    if re.search(r"\bDELETE\b|\bUPDATE\b", sql, re.IGNORECASE) and \
       not re.search(r"\bWHERE\b", sql, re.IGNORECASE):
        findings.append({"rule_id": "R002/R003", "vuln_class": "DML_NO_WHERE", "risk": 9})
    if re.search(r"SELECT\s+\*", sql, re.IGNORECASE):
        findings.append({"rule_id": "R001", "vuln_class": "SELECT_STAR", "risk": 5})
    # Прямой regex на имена sensitive — limited
    for col in re.findall(r"\b(password|passport|card_number|ssn)\b", sql, re.IGNORECASE):
        findings.append({"rule_id": "R009", "vuln_class": "DIRECT_SENSITIVE", "risk": 7})
    return findings


section("Judge A (только правила) — прогон")
for c in CASES:
    f = judge_A_rules_only(c["sql"])
    pred_vuln = bool(f)
    correct = pred_vuln == c["ground_truth_vuln"]
    mark = "✅" if correct else "❌"
    print(f"  {mark} {c['id']:35s}  pred={pred_vuln}  truth={c['ground_truth_vuln']}  findings={len(f)}")


## 3. Judge B — только LLM (mock, имитирует inconsistency)


In [ ]:
import random as _rnd


##
# @brief Mock LLM-only judge — имитирует, что LLM «угадывает» по контексту,
#        но без structured findings часто ошибается.
def judge_B_llm_only(sql, seed=None):
    rng = _rnd.Random(seed)
    # Реальный LLM на голом SQL без RAG/findings даёт ~70% точности
    # с inconsistency между прогонами. Симулируем шум.
    sql_lo = sql.lower()
    score = 0.0
    if "pg_sleep" in sql_lo and "case" in sql_lo:
        score = 0.8
    elif "pg_sleep" in sql_lo:
        score = 0.5  # не различает blind vs миграцию
    elif "delete" in sql_lo and "where" not in sql_lo:
        score = 0.9
    elif "select *" in sql_lo:
        score = 0.4
    elif "permissions" in sql_lo:
        score = 0.3  # не понимает семантику JSON прав
    score += rng.uniform(-0.2, 0.2)  # inconsistency
    return score > 0.5


section("Judge B (только LLM) — 3 прогона на каждом кейсе (для inconsistency)")
for c in CASES:
    runs = [judge_B_llm_only(c["sql"], seed=i) for i in range(3)]
    consistent = len(set(runs)) == 1
    mark = "✅" if all(r == c["ground_truth_vuln"] for r in runs) else "❌"
    print(f"  {mark} {c['id']:35s}  прогоны={runs}  consistent={consistent}")


## 4. Judge C — гибрид (Phase 1 → Phase 2 триаж + RAG)


In [ ]:
##
# @brief Mock RAG — словарь CWE/CAPEC-чанков.
RAG_KB = {
    "R001":      {"cwe_id": "CWE-1295", "doc": "SELECT * раскрывает все колонки, включая возможно чувствительные."},
    "R002/R003": {"cwe_id": "CWE-1284", "doc": "UPDATE/DELETE без WHERE затрагивает все строки таблицы."},
    "R006-pg-sleep": {"cwe_id": "CWE-89", "capec_id": "CAPEC-7", "doc": "pg_sleep в условии CASE — индикатор blind SQLi."},
    "R009":      {"cwe_id": "CWE-200", "doc": "Прямой доступ к ПДн без маскирования."},
}


##
# @brief Phase 2 — LLM-триаж findings с RAG. Mock через rule-based решения.
def phase2_triage(sql, findings, context_note=""):
    """@brief Триажит каждый finding, отсеивает FP по контексту, добавляет evidence."""
    triaged = []
    for f in findings:
        # FP-фильтр: если в SQL есть DDL/комментарий «миграция» и pg_sleep — отбрасываем
        if f["rule_id"] == "R006-pg-sleep":
            if "ALTER TABLE" in sql.upper() or "миграция" in sql.lower():
                continue  # FP — legitimate
            # CASE WHEN ... pg_sleep — это blind, повышаем risk
            if "CASE" in sql.upper() and "WHEN" in sql.upper():
                f["risk"] = 9
        # FP-фильтр для DML без WHERE: TRUNCATE-like → low
        if f["rule_id"] == "R002/R003" and re.search(r"--.*очист", sql, re.IGNORECASE):
            f["risk"] = 2
        rag = RAG_KB.get(f["rule_id"], {})
        f["evidence"] = rag
        triaged.append(f)

    # Phase 2 ищет дополнительные семантические уязвимости, которые Phase 1 пропустил
    # (mock: ищем JSON-permissions через имя)
    if "permissions" in sql.lower() and not any("R009" in t["rule_id"] for t in triaged):
        triaged.append({
            "rule_id":     "R009-semantic",
            "vuln_class":  "DIRECT_SENSITIVE",
            "risk":        7,
            "evidence":    {"cwe_id": "CWE-200", "doc": "permissions = JSON прав, утечка"},
            "_from":       "phase2-semantic",
        })
    return triaged


def judge_C_hybrid(sql, context_note=""):
    phase1 = judge_A_rules_only(sql)
    return phase2_triage(sql, phase1, context_note)


section("Judge C (гибрид) — прогон")
for c in CASES:
    f = judge_C_hybrid(c["sql"], c["context_note"])
    pred_vuln = bool(f)
    correct = pred_vuln == c["ground_truth_vuln"]
    mark = "✅" if correct else "❌"
    print(f"  {mark} {c['id']:35s}  pred={pred_vuln}  truth={c['ground_truth_vuln']}")
    for finding in f:
        evidence = finding.get("evidence", {})
        cwe = evidence.get("cwe_id", "—")
        print(f"        - {finding['rule_id']:15s} risk={finding['risk']}  CWE={cwe}")


## 5. Сравнение Precision / Recall


In [ ]:
def metrics(judge_fn, name):
    tp = fp = tn = fn = 0
    for c in CASES:
        try:
            pred = bool(judge_fn(c["sql"], c.get("context_note", "")))
        except TypeError:
            pred = bool(judge_fn(c["sql"]))
        truth = c["ground_truth_vuln"]
        if pred and truth:    tp += 1
        elif pred and not truth: fp += 1
        elif not pred and truth: fn += 1
        else: tn += 1
    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0
    return name, p, r, {"tp": tp, "fp": fp, "tn": tn, "fn": fn}


section("Финальное сравнение")
print(f"{'судья':<22} {'precision':>10} {'recall':>10}   counts")
print("-" * 72)
for fn, label in [(judge_A_rules_only, "A — правила"),
                  (judge_B_llm_only,   "B — LLM-only"),
                  (judge_C_hybrid,     "C — гибрид")]:
    n, p, r, c = metrics(fn, label)
    print(f"{n:<22} {p:>10.2f} {r:>10.2f}   {c}")


## Итог

Мы увидели проблему **под микроскопом** и **симуляцию решения** из ADR.

## Куда дальше

- **Описание проблемы:** [problems/engineering/04-llm-judge-unreliability/README.md](../../problems/engineering/04-llm-judge-unreliability/README.md)
- **Варианты решения + почему так:** [problems/engineering/04-llm-judge-unreliability/solutions.md](../../problems/engineering/04-llm-judge-unreliability/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../../docs/adr/0002-loop-architecture-langgraph.md)
